In [1]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

np.random.seed(42)
random.seed(42)

In [2]:
import random

# NO seed set
print(random.randint(1, 100))
print(random.randint(1, 100))

82
15


In [3]:
import random

random.seed(40)

print(random.randint(1, 100))
print(random.randint(1, 100))

59
75


In [4]:
conn = sqlite3.connect('access_logs.db')
cursor = conn.cursor()

In [5]:
cursor.executescript('''
DROP TABLE IF EXISTS employees;
DROP TABLE IF EXISTS login_events;
DROP TABLE IF EXISTS file_access_events;

CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    role TEXT,
    typical_start_hour INTEGER,
    typical_end_hour INTEGER
);

CREATE TABLE login_events (
    login_id INTEGER PRIMARY KEY AUTOINCREMENT,
    employee_id INTEGER,
    login_timestamp TEXT,
    ip_address TEXT,
    success INTEGER,
    FOREIGN KEY (employee_id) REFERENCES employees(employee_id)
);

CREATE TABLE file_access_events (
    access_id INTEGER PRIMARY KEY AUTOINCREMENT,
    employee_id INTEGER,
    file_id INTEGER,
    access_timestamp TEXT,
    action TEXT,
    sensitivity_level TEXT,
    FOREIGN KEY (employee_id) REFERENCES employees(employee_id)
);
''')
conn.commit()

In [6]:
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)

,name
0,sqlite_sequence
1,session_labels
2,employees
3,login_events
4,file_access_events


In [7]:
departments = ['Engineering', 'Finance', 'HR', 'Marketing', 'Sales', 'IT']
roles = ['Analyst', 'Manager', 'Associate', 'Senior Associate', 'Director']

first_names = ['James','Mary','John','Patricia','Robert','Jennifer','Michael','Linda',
               'William','Elizabeth','David','Barbara','Richard','Susan','Joseph','Jessica',
               'Thomas','Sarah','Charles','Karen','Daniel','Nancy','Matthew','Lisa','Anthony']
last_names = ['Smith','Johnson','Williams','Brown','Jones','Garcia','Miller','Davis',
              'Rodriguez','Martinez','Hernandez','Lopez','Gonzalez','Wilson','Anderson',
              'Thomas','Taylor','Moore','Jackson','Martin']

employees = []
for emp_id in range(1, 51):  # 50 employees
    name = f"{random.choice(first_names)} {random.choice(last_names)}"
    dept = random.choice(departments)
    role = random.choice(roles)
    
    # Most employees work standard hours with slight individual variation
    start_hour = random.choice([8, 9, 9, 9, 10])   # 9am is most common
    end_hour = start_hour + random.choice([7, 8, 8, 9])
    
    employees.append((emp_id, name, dept, role, start_hour, end_hour))

cursor.executemany(
    "INSERT INTO employees VALUES (?, ?, ?, ?, ?, ?)", employees
)
conn.commit()

pd.read_sql_query("SELECT * FROM employees LIMIT 10", conn)

,employee_id,name,department,role,typical_start_hour,typical_end_hour
0,1,Thomas Johnson,Finance,Associate,9,17
1,2,Anthony Lopez,HR,Senior Associate,8,16
2,3,Mary Miller,Sales,Senior Associate,8,16
3,4,Karen Hernandez,Engineering,Associate,9,17
4,5,Thomas Taylor,Engineering,Senior Associate,8,16
5,6,Lisa Hernandez,Finance,Manager,9,17
6,7,Thomas Martinez,IT,Associate,9,17
7,8,Charles Miller,Marketing,Senior Associate,10,17
8,9,Jennifer Jackson,Finance,Director,10,17
9,10,John Anderson,Sales,Associate,8,16


In [8]:
pd.read_sql_query("SELECT COUNT(*) FROM employees", conn)

,COUNT(*)
0,50


In [9]:
pd.read_sql_query("SELECT department, COUNT(*) FROM employees GROUP BY department", conn)

,department,COUNT(*)
0,Engineering,7
1,Finance,9
2,HR,9
3,IT,5
4,Marketing,10
5,Sales,10


In [10]:
pd.read_sql_query("SELECT MIN(typical_start_hour), MAX(typical_end_hour)  FROM employees", conn)

,MIN(typical_start_hour),MAX(typical_end_hour)
0,8,18


In [11]:
start_date = datetime(2026, 1, 1)
num_days = 90

login_events = []

for emp in employees:
    emp_id, name, dept, role, start_hour, end_hour = emp
    
    # NEW: assign this employee 2-3 "usual" IPs ONCE, before looping through days
    num_usual_ips = random.randint(2, 3)
    employee_usual_ips = [
        f"10.0.{random.randint(1,20)}.{random.randint(1,254)}" 
        for _ in range(num_usual_ips)
    ]
    
    for day_offset in range(num_days):
        current_date = start_date + timedelta(days=day_offset)
        
        if current_date.weekday() >= 5:
            continue
        
        if random.random() > 0.9:
            continue
        
        login_hour = int(np.clip(np.random.normal(start_hour, 0.75), start_hour - 2, start_hour + 3))
        login_minute = random.randint(0, 59)
        
        login_time = current_date.replace(hour=login_hour, minute=login_minute, second=0)
        
        # CHANGED: pick from this employee's usual IPs, instead of a fresh random one every time
        ip_address = random.choice(employee_usual_ips)
        
        login_events.append((emp_id, login_time.isoformat(), ip_address, 1))

cursor.executemany(
    "INSERT INTO login_events (employee_id, login_timestamp, ip_address, success) VALUES (?, ?, ?, ?)",
    login_events
)
conn.commit()

print(f"Generated {len(login_events)} normal login events")
pd.read_sql_query("SELECT * FROM login_events LIMIT 10", conn)

Generated 2905 normal login events


,login_id,employee_id,login_timestamp,ip_address,success
0,1,1,2026-01-01T09:01:00,10.0.4.238,1
1,2,1,2026-01-02T08:23:00,10.0.4.238,1
2,3,1,2026-01-05T09:46:00,10.0.14.22,1
3,4,1,2026-01-07T10:46:00,10.0.14.22,1
4,5,1,2026-01-08T08:20:00,10.0.14.22,1
5,6,1,2026-01-09T08:56:00,10.0.4.238,1
6,7,1,2026-01-12T10:54:00,10.0.4.238,1
7,8,1,2026-01-14T09:21:00,10.0.14.22,1
8,9,1,2026-01-15T08:55:00,10.0.14.22,1
9,10,1,2026-01-16T09:11:00,10.0.4.238,1


In [12]:
print(len(login_events))

2905


In [13]:
check_df = pd.read_sql_query("SELECT login_timestamp FROM login_events LIMIT 50", conn)
print(check_df)


        login_timestamp
0   2026-01-01T09:01:00
1   2026-01-02T08:23:00
2   2026-01-05T09:46:00
3   2026-01-07T10:46:00
4   2026-01-08T08:20:00
5   2026-01-09T08:56:00
6   2026-01-12T10:54:00
7   2026-01-14T09:21:00
8   2026-01-15T08:55:00
9   2026-01-16T09:11:00
10  2026-01-19T08:34:00
11  2026-01-20T08:37:00
12  2026-01-22T09:37:00
13  2026-01-23T07:54:00
14  2026-01-26T07:11:00
15  2026-01-27T08:10:00
16  2026-01-28T08:20:00
17  2026-01-29T09:52:00
18  2026-01-30T08:28:00
19  2026-02-02T07:17:00
20  2026-02-03T10:23:00
21  2026-02-04T08:54:00
22  2026-02-05T09:46:00
23  2026-02-06T07:55:00
24  2026-02-09T08:59:00
25  2026-02-11T09:35:00
26  2026-02-12T08:33:00
27  2026-02-13T09:23:00
28  2026-02-16T08:10:00
29  2026-02-17T08:56:00
30  2026-02-18T08:34:00
31  2026-02-19T10:23:00
32  2026-02-20T08:54:00
33  2026-02-24T08:03:00
34  2026-02-25T09:19:00
35  2026-02-26T08:36:00
36  2026-03-02T09:34:00
37  2026-03-03T07:03:00
38  2026-03-04T08:14:00
39  2026-03-05T09:31:00
40  2026-03-06T0

In [14]:
pd.read_sql_query("SELECT * FROM login_events WHERE login_timestamp LIKE '2026-03-04%'", conn)

,login_id,employee_id,login_timestamp,ip_address,success
0,39,1,2026-03-04T08:14:00,10.0.4.238,1
1,99,2,2026-03-04T08:07:00,10.0.10.192,1
2,157,3,2026-03-04T09:13:00,10.0.17.197,1
3,276,5,2026-03-04T08:28:00,10.0.1.161,1
4,391,7,2026-03-04T08:51:00,10.0.18.110,1
5,450,8,2026-03-04T09:22:00,10.0.17.58,1
6,509,9,2026-03-04T10:29:00,10.0.1.135,1
7,568,10,2026-03-04T07:40:00,10.0.2.67,1
8,629,11,2026-03-04T09:47:00,10.0.3.121,1
9,684,12,2026-03-04T10:29:00,10.0.16.141,1


In [15]:

test_skips = [random.random() > 0.9 for _ in range(10000)]
print(sum(test_skips) / len(test_skips))  

0.1028


In [16]:
raw = np.random.normal(9, 0.75)
print(raw)  
clipped = np.clip(raw, 7, 11)
print(clipped)  
print(int(clipped))

9.562934375641325
9.562934375641325
9


In [17]:
file_access_events = []

actions = ['read', 'read', 'read', 'write', 'write', 'delete']  # read is most common
sensitivity_levels = ['low', 'low', 'low', 'medium', 'medium', 'high']  

for emp_id, login_ts_str, ip, success in login_events:
    login_ts = datetime.fromisoformat(login_ts_str)
    num_files_accessed = random.randint(2, 6)
    for x in range(num_files_accessed):
        minutes_after_login = random.randint(1, 360)
        access_time = login_ts + timedelta(minutes=minutes_after_login)
        file_id = random.randint(1, 500) 
        action = random.choice(actions)
        sensitivity = random.choice(sensitivity_levels)
        file_access_events.append((emp_id, file_id, access_time.isoformat(), action, sensitivity))

cursor.executemany(
    "INSERT INTO file_access_events (employee_id, file_id, access_timestamp, action, sensitivity_level) VALUES (?, ?, ?, ?, ?)",
    file_access_events
)
conn.commit()

print(f"Generated {len(file_access_events)} normal file access events")
pd.read_sql_query("SELECT * FROM file_access_events LIMIT 10", conn)

Generated 11528 normal file access events


,access_id,employee_id,file_id,access_timestamp,action,sensitivity_level
0,1,1,273,2026-01-01T14:52:00,read,low
1,2,1,163,2026-01-01T11:55:00,write,low
2,3,1,500,2026-01-01T13:55:00,write,high
3,4,1,311,2026-01-01T13:47:00,read,high
4,5,1,329,2026-01-02T09:24:00,write,high
5,6,1,179,2026-01-02T14:23:00,write,medium
6,7,1,39,2026-01-02T13:20:00,write,high
7,8,1,318,2026-01-02T09:58:00,read,low
8,9,1,61,2026-01-02T14:11:00,read,medium
9,10,1,146,2026-01-05T10:54:00,read,medium


In [18]:
check = pd.read_sql_query('''
    SELECT f.access_timestamp, l.login_timestamp
    FROM file_access_events f
    JOIN login_events l ON f.employee_id = l.employee_id
    WHERE f.access_timestamp < l.login_timestamp
    LIMIT 5
''', conn)
print(check)

      access_timestamp      login_timestamp
0  2026-01-01T14:52:00  2026-01-02T08:23:00
1  2026-01-01T14:52:00  2026-01-05T09:46:00
2  2026-01-01T14:52:00  2026-01-07T10:46:00
3  2026-01-01T14:52:00  2026-01-08T08:20:00
4  2026-01-01T14:52:00  2026-01-09T08:56:00


In [19]:
print(len(file_access_events) / len(login_events))

3.968330464716007


In [20]:
cursor.executescript('''
DROP TABLE IF EXISTS session_labels;

CREATE TABLE session_labels (
    label_id INTEGER PRIMARY KEY AUTOINCREMENT,
    employee_id INTEGER,
    session_date TEXT,
    is_anomaly INTEGER,
    anomaly_type TEXT
);
''')
conn.commit()

In [21]:
anomaly_types = ['off_hours', 'volume_spike', 'sensitivity_spike', 'unfamiliar_ip']

num_anomalies = int(len(login_events) * 0.03)  # ~3% of sessions
anomaly_sample = random.sample(login_events, num_anomalies)

session_labels = []

for emp_id, login_ts_str, ip, success in anomaly_sample:
    login_ts = datetime.fromisoformat(login_ts_str)
    anomaly_type = random.choice(anomaly_types)
    
    if anomaly_type == 'off_hours':
        # Force login time to 1am-4am
        odd_hour = random.randint(1, 4)
        new_login_ts = login_ts.replace(hour=odd_hour, minute=random.randint(0,59))
        
        cursor.execute(
            "UPDATE login_events SET login_timestamp = ? WHERE employee_id = ? AND login_timestamp = ?",
            (new_login_ts.isoformat(), emp_id, login_ts_str)
        )
        session_date = new_login_ts.date().isoformat()

    elif anomaly_type == 'volume_spike':
        # Add a burst of extra file accesses (15-30 files) right after this login
        num_extra_files = random.randint(15, 30)
        for _ in range(num_extra_files):
            minutes_after = random.randint(1, 120)
            access_time = login_ts + timedelta(minutes=minutes_after)
            file_id = random.randint(1, 500)
            action = random.choice(['read', 'write', 'read'])
            sensitivity = random.choice(['low', 'medium'])
            cursor.execute(
                "INSERT INTO file_access_events (employee_id, file_id, access_timestamp, action, sensitivity_level) VALUES (?, ?, ?, ?, ?)",
                (emp_id, file_id, access_time.isoformat(), action, sensitivity)
            )
        session_date = login_ts.date().isoformat()

    elif anomaly_type == 'sensitivity_spike':
        # Add several HIGH sensitivity file accesses
        num_high_files = random.randint(5, 10)
        for _ in range(num_high_files):
            minutes_after = random.randint(1, 120)
            access_time = login_ts + timedelta(minutes=minutes_after)
            file_id = random.randint(1, 500)
            cursor.execute(
                "INSERT INTO file_access_events (employee_id, file_id, access_timestamp, action, sensitivity_level) VALUES (?, ?, ?, ?, ?)",
                (emp_id, file_id, access_time.isoformat(), 'read', 'high')
            )
        session_date = login_ts.date().isoformat()

    elif anomaly_type == 'unfamiliar_ip':
        # Use an external-looking, unfamiliar IP
        odd_ip = f"{random.randint(100,200)}.{random.randint(1,254)}.{random.randint(1,254)}.{random.randint(1,254)}"
        cursor.execute(
            "UPDATE login_events SET ip_address = ? WHERE employee_id = ? AND login_timestamp = ?",
            (odd_ip, emp_id, login_ts_str)
        )
        session_date = login_ts.date().isoformat()

    session_labels.append((emp_id, session_date, 1, anomaly_type))

cursor.executemany(
    "INSERT INTO session_labels (employee_id, session_date, is_anomaly, anomaly_type) VALUES (?, ?, ?, ?)",
    session_labels
)
conn.commit()

print(f"Injected {len(session_labels)} anomalous sessions")
pd.read_sql_query("SELECT anomaly_type, COUNT(*) as count FROM session_labels GROUP BY anomaly_type", conn)

Injected 87 anomalous sessions


,anomaly_type,count
0,off_hours,24
1,sensitivity_spike,15
2,unfamiliar_ip,19
3,volume_spike,29


In [27]:
sample_anomaly = pd.read_sql_query("SELECT * FROM session_labels WHERE anomaly_type='unfamiliar_ip' LIMIT 1", conn)
sample_anomaly
emp_check = sample_anomaly.iloc[0]['employee_id']
date_check = sample_anomaly.iloc[0]['session_date']
pd.read_sql_query(f"SELECT COUNT(*) FROM file_access_events WHERE employee_id={emp_check} AND DATE(access_timestamp)='{date_check}'", conn)

,COUNT(*)
0,5


In [22]:
before_count = pd.read_sql_query("SELECT COUNT(*) as cnt FROM login_events", conn)
print(before_count)

    cnt
0  2905


In [23]:
feature_query = '''
WITH login_with_emp AS (
    SELECT 
        l.login_id,
        l.employee_id,
        l.login_timestamp,
        DATE(l.login_timestamp) AS login_date,
        CAST(strftime('%H', l.login_timestamp) AS INTEGER) AS login_hour,
        l.ip_address,
        e.typical_start_hour,
        e.typical_end_hour
    FROM login_events l
    JOIN employees e ON l.employee_id = e.employee_id
),

off_hours_flag AS (
    SELECT *,
        CASE 
            WHEN login_hour < typical_start_hour - 1 OR login_hour > typical_end_hour + 1 
            THEN 1 ELSE 0 
        END AS is_off_hours
    FROM login_with_emp
),

ip_history AS (
    SELECT *,
        CASE
            WHEN ip_address IN (
                SELECT ip_address FROM login_events l2
                WHERE l2.employee_id = off_hours_flag.employee_id
                AND l2.login_timestamp < off_hours_flag.login_timestamp
            ) THEN 0 ELSE 1
        END AS is_new_ip
    FROM off_hours_flag
),

login_gaps AS (
    SELECT *,
        LAG(login_timestamp) OVER (
            PARTITION BY employee_id ORDER BY login_timestamp
        ) AS prev_login_timestamp
    FROM ip_history
),

with_gap_days AS (
    SELECT *,
        CASE 
            WHEN prev_login_timestamp IS NULL THEN NULL
            ELSE (julianday(login_timestamp) - julianday(prev_login_timestamp))
        END AS days_since_last_login
    FROM login_gaps
),

file_stats AS (
    SELECT 
        employee_id,
        DATE(access_timestamp) AS access_date,
        COUNT(*) AS file_access_count,
        COUNT(DISTINCT file_id) AS distinct_files_count,
        SUM(CASE WHEN sensitivity_level = 'high' THEN 1 ELSE 0 END) AS high_sensitivity_count,
        SUM(CASE WHEN action = 'delete' THEN 1 ELSE 0 END) AS delete_count
    FROM file_access_events
    GROUP BY employee_id, DATE(access_timestamp)
)

SELECT 
    w.login_id,
    w.employee_id,
    w.login_date,
    w.login_hour,
    w.is_off_hours,
    w.is_new_ip,
    w.days_since_last_login,
    COALESCE(f.file_access_count, 0) AS file_access_count,
    COALESCE(f.distinct_files_count, 0) AS distinct_files_count,
    COALESCE(f.high_sensitivity_count, 0) AS high_sensitivity_count,
    COALESCE(f.delete_count, 0) AS delete_count
FROM with_gap_days w
LEFT JOIN file_stats f 
    ON w.employee_id = f.employee_id AND w.login_date = f.access_date
ORDER BY w.employee_id, w.login_date
'''

features_df = pd.read_sql_query(feature_query, conn)
print(features_df.shape)
features_df.head(100)

(2905, 11)


,login_id,employee_id,login_date,login_hour,is_off_hours,is_new_ip,days_since_last_login,file_access_count,distinct_files_count,high_sensitivity_count,delete_count
0,1,1,2026-01-01,9,0,1,NaN,4,4,2,0
1,2,1,2026-01-02,8,0,0,0.973611,5,5,2,0
2,3,1,2026-01-05,9,0,1,3.057639,6,5,0,1
3,4,1,2026-01-07,10,0,0,2.041667,6,6,1,2
4,5,1,2026-01-08,8,0,0,0.898611,2,2,0,0
...,...,...,...,...,...,...,...,...,...,...,...
95,96,2,2026-02-27,6,1,0,0.940972,3,3,1,0
96,97,2,2026-03-02,8,0,0,3.092361,29,28,1,1
97,98,2,2026-03-03,8,0,0,1.016667,4,4,1,1
98,99,2,2026-03-04,8,0,0,0.968750,2,2,0,1


In [24]:
test3 = pd.read_sql_query('''
WITH login_with_emp AS (
    SELECT 
        l.login_id, l.employee_id, l.login_timestamp,
        DATE(l.login_timestamp) AS login_date,
        CAST(strftime('%H', l.login_timestamp) AS INTEGER) AS login_hour,
        l.ip_address, e.typical_start_hour, e.typical_end_hour
    FROM login_events l
    JOIN employees e ON l.employee_id = e.employee_id
),
off_hours_flag AS (
    SELECT *,
        CASE WHEN login_hour < typical_start_hour - 1 OR login_hour > typical_end_hour + 1 
        THEN 1 ELSE 0 END AS is_off_hours
    FROM login_with_emp
),
ip_history AS (
    SELECT *,
        CASE
            WHEN ip_address IN (
                SELECT ip_address FROM login_events l2
                WHERE l2.employee_id = off_hours_flag.employee_id
                AND l2.login_timestamp < off_hours_flag.login_timestamp
            ) THEN 0 ELSE 1
        END AS is_new_ip
    FROM off_hours_flag
),
login_gaps AS (
    SELECT *,
        LAG(login_timestamp) OVER (PARTITION BY employee_id ORDER BY login_timestamp) AS prev_login_timestamp
    FROM ip_history
)
SELECT employee_id, login_timestamp, prev_login_timestamp, ip_address, is_new_ip 
FROM login_gaps 
WHERE employee_id = 1
ORDER BY login_timestamp
LIMIT 150
''', conn)
test3

,employee_id,login_timestamp,prev_login_timestamp,ip_address,is_new_ip
0,1,2026-01-01T09:01:00,NaN,10.0.4.238,1
1,1,2026-01-02T08:23:00,2026-01-01T09:01:00,10.0.4.238,0
2,1,2026-01-05T09:46:00,2026-01-02T08:23:00,10.0.14.22,1
3,1,2026-01-07T10:46:00,2026-01-05T09:46:00,10.0.14.22,0
4,1,2026-01-08T08:20:00,2026-01-07T10:46:00,10.0.14.22,0
5,1,2026-01-09T08:56:00,2026-01-08T08:20:00,10.0.4.238,0
6,1,2026-01-12T10:54:00,2026-01-09T08:56:00,10.0.4.238,0
7,1,2026-01-14T09:21:00,2026-01-12T10:54:00,10.0.14.22,0
8,1,2026-01-15T08:55:00,2026-01-14T09:21:00,10.0.14.22,0
9,1,2026-01-16T09:11:00,2026-01-15T08:55:00,10.0.4.238,0


In [25]:
# check: how often does employee 1 actually reuse the SAME exact IP across different logins?
pd.read_sql_query('''
    SELECT ip_address, COUNT(*) as times_used
    FROM login_events
    WHERE employee_id = 1
    GROUP BY ip_address
    ORDER BY times_used DESC
    LIMIT 50
    
''', conn)

,ip_address,times_used
0,10.0.4.238,31
1,10.0.14.22,25
2,129.215.79.234,1


In [26]:
test5 = pd.read_sql_query('''
WITH login_with_emp AS (
    SELECT l.login_id, l.employee_id, l.login_timestamp,
        DATE(l.login_timestamp) AS login_date,
        CAST(strftime('%H', l.login_timestamp) AS INTEGER) AS login_hour,
        l.ip_address, e.typical_start_hour, e.typical_end_hour
    FROM login_events l
    JOIN employees e ON l.employee_id = e.employee_id
),
off_hours_flag AS (
    SELECT *,
        CASE WHEN login_hour < typical_start_hour - 1 OR login_hour > typical_end_hour + 1 
        THEN 1 ELSE 0 END AS is_off_hours
    FROM login_with_emp
),
ip_history AS (
    SELECT *,
        CASE
            WHEN ip_address IN (
                SELECT ip_address FROM login_events l2
                WHERE l2.employee_id = off_hours_flag.employee_id
                AND l2.login_timestamp < off_hours_flag.login_timestamp
            ) THEN 0 ELSE 1
        END AS is_new_ip
    FROM off_hours_flag
),
login_gaps AS (
    SELECT *,
        LAG(login_timestamp) OVER (PARTITION BY employee_id ORDER BY login_timestamp) AS prev_login_timestamp
    FROM ip_history
),
with_gap_days AS (
    SELECT *,
        CASE 
            WHEN prev_login_timestamp IS NULL THEN NULL
            ELSE (julianday(login_timestamp) - julianday(prev_login_timestamp))
        END AS days_since_last_login
    FROM login_gaps
)
SELECT employee_id, login_timestamp, prev_login_timestamp, days_since_last_login
FROM with_gap_days
WHERE employee_id = 1
ORDER BY login_timestamp
LIMIT 15
''', conn)
test5

,employee_id,login_timestamp,prev_login_timestamp,days_since_last_login
0,1,2026-01-01T09:01:00,NaN,NaN
1,1,2026-01-02T08:23:00,2026-01-01T09:01:00,0.973611
2,1,2026-01-05T09:46:00,2026-01-02T08:23:00,3.057639
3,1,2026-01-07T10:46:00,2026-01-05T09:46:00,2.041667
4,1,2026-01-08T08:20:00,2026-01-07T10:46:00,0.898611
5,1,2026-01-09T08:56:00,2026-01-08T08:20:00,1.025000
6,1,2026-01-12T10:54:00,2026-01-09T08:56:00,3.081944
7,1,2026-01-14T09:21:00,2026-01-12T10:54:00,1.935417
8,1,2026-01-15T08:55:00,2026-01-14T09:21:00,0.981944
9,1,2026-01-16T09:11:00,2026-01-15T08:55:00,1.011111


In [27]:
# distinct_files_count should NEVER exceed file_access_count - confirm this holds for ALL rows, not just employee 1
pd.read_sql_query('''
    SELECT COUNT(*) as violations
    FROM (
        SELECT 
            employee_id, DATE(access_timestamp) AS access_date,
            COUNT(*) AS file_access_count,
            COUNT(DISTINCT file_id) AS distinct_files_count
        FROM file_access_events
        GROUP BY employee_id, DATE(access_timestamp)
    )
    WHERE distinct_files_count > file_access_count
''', conn)

,violations
0,0


In [28]:
test6 = pd.read_sql_query('''
WITH file_stats AS (
    SELECT 
        employee_id,
        DATE(access_timestamp) AS access_date,
        COUNT(*) AS file_access_count,
        COUNT(DISTINCT file_id) AS distinct_files_count,
        SUM(CASE WHEN sensitivity_level = 'high' THEN 1 ELSE 0 END) AS high_sensitivity_count,
        SUM(CASE WHEN action = 'delete' THEN 1 ELSE 0 END) AS delete_count
    FROM file_access_events
    GROUP BY employee_id, DATE(access_timestamp)
)
SELECT * FROM file_stats
WHERE employee_id = 1
ORDER BY access_date
LIMIT 15
''', conn)
test6

,employee_id,access_date,file_access_count,distinct_files_count,high_sensitivity_count,delete_count
0,1,2026-01-01,4,4,2,0
1,1,2026-01-02,5,5,2,0
2,1,2026-01-05,6,5,0,1
3,1,2026-01-07,6,6,1,2
4,1,2026-01-08,2,2,0,0
5,1,2026-01-09,4,4,0,0
6,1,2026-01-12,6,6,2,2
7,1,2026-01-14,2,2,0,0
8,1,2026-01-15,3,3,0,1
9,1,2026-01-16,2,2,0,0


In [29]:
check_query = '''
WITH login_with_emp AS (
    SELECT l.login_id, l.employee_id, l.login_timestamp, l.ip_address
    FROM login_events l
),
first_logins AS (
    SELECT employee_id, MIN(login_timestamp) as first_login
    FROM login_events
    GROUP BY employee_id
)
SELECT l.employee_id, l.login_timestamp, l.ip_address
FROM login_events l
JOIN first_logins f 
    ON l.employee_id = f.employee_id 
    AND l.login_timestamp = f.first_login
LIMIT 5
'''
pd.read_sql_query(check_query, conn)

,employee_id,login_timestamp,ip_address
0,1,2026-01-01T09:01:00,10.0.4.238
1,2,2026-01-01T07:26:00,10.0.5.124
2,3,2026-01-01T07:08:00,10.0.16.189
3,4,2026-01-01T09:43:00,10.0.7.120
4,5,2026-01-01T06:12:00,10.0.6.245


In [30]:
pd.read_sql_query('''
    SELECT COUNT(*) FROM login_events l
    JOIN employees e ON l.employee_id = e.employee_id
    WHERE CAST(strftime('%H', l.login_timestamp) AS INTEGER) < e.typical_start_hour - 1
       OR CAST(strftime('%H', l.login_timestamp) AS INTEGER) > e.typical_end_hour + 1
''', conn)

,COUNT(*)
0,264


In [31]:
labels_df = pd.read_sql_query("SELECT * FROM session_labels", conn)

# Merge on employee_id + date
features_df = features_df.merge(
    labels_df[['employee_id', 'session_date', 'is_anomaly', 'anomaly_type']],
    left_on=['employee_id', 'login_date'],
    right_on=['employee_id', 'session_date'],
    how='left'
)

features_df['is_anomaly'] = features_df['is_anomaly'].fillna(0).astype(int)
features_df = features_df.drop(columns=['session_date'])
print(features_df['is_anomaly'].sum()) 
print(features_df['is_anomaly'].value_counts())
features_df.head()

87
is_anomaly
0    2818
1      87
Name: count, dtype: int64


,login_id,employee_id,login_date,login_hour,is_off_hours,is_new_ip,days_since_last_login,file_access_count,distinct_files_count,high_sensitivity_count,delete_count,is_anomaly,anomaly_type
0,1,1,2026-01-01,9,0,1,NaN,4,4,2,0,0,NaN
1,2,1,2026-01-02,8,0,0,0.973611,5,5,2,0,0,NaN
2,3,1,2026-01-05,9,0,1,3.057639,6,5,0,1,0,NaN
3,4,1,2026-01-07,10,0,0,2.041667,6,6,1,2,0,NaN
4,5,1,2026-01-08,8,0,0,0.898611,2,2,0,0,0,NaN


In [32]:
# spot check: pick one known anomaly and confirm it merged correctly
sample = labels_df.iloc[0]
check = features_df[
    (features_df['employee_id'] == sample['employee_id']) & 
    (features_df['login_date'] == sample['session_date'])
]
print(check[['employee_id', 'login_date', 'is_anomaly', 'anomaly_type']])

      employee_id  login_date  is_anomaly       anomaly_type
1072           19  2026-02-23           1  sensitivity_spike


In [33]:
# Fill first-ever login gap with the median gap (reasonable default, not 0)
median_gap = features_df['days_since_last_login'].median()
features_df['days_since_last_login'] = features_df['days_since_last_login'].fillna(median_gap)

# Final feature columns we'll feed to the models
feature_cols = [
    'login_hour', 'is_off_hours', 'is_new_ip', 'days_since_last_login',
    'file_access_count', 'distinct_files_count', 'high_sensitivity_count', 'delete_count'
]

X = features_df[feature_cols]
y = features_df['is_anomaly']

print(X.isnull().sum())  # should all be 0
X.describe()

login_hour                0
is_off_hours              0
is_new_ip                 0
days_since_last_login     0
file_access_count         0
distinct_files_count      0
high_sensitivity_count    0
delete_count              0
dtype: int64


,login_hour,is_off_hours,is_new_ip,days_since_last_login,file_access_count,distinct_files_count,high_sensitivity_count,delete_count
count,2905.000000,2905.000000,2905.000000,2905.000000,2905.000000,2905.000000,2905.000000,2905.000000
mean,8.411704,0.090878,0.048881,1.545394,4.239931,4.216523,0.695009,0.666781
std,1.165033,0.287485,0.215657,0.937492,2.820244,2.759083,0.931869,0.781467
min,1.000000,0.000000,0.000000,0.604861,2.000000,1.000000,0.000000,0.000000
25%,8.000000,0.000000,0.000000,0.981944,3.000000,3.000000,0.000000,0.000000
50%,8.000000,0.000000,0.000000,1.027083,4.000000,4.000000,1.000000,1.000000
75%,9.000000,0.000000,0.000000,2.013194,5.000000,5.000000,1.000000,1.000000
max,12.000000,1.000000,1.000000,5.102083,36.000000,36.000000,12.000000,4.000000


In [34]:
print(X.shape, y.shape)


(2905, 8) (2905,)


In [35]:
print(f"Median gap: {median_gap}")
print(features_df['days_since_last_login'].describe())

Median gap: 1.0270833331160247
count    2905.000000
mean        1.545394
std         0.937492
min         0.604861
25%         0.981944
50%         1.027083
75%         2.013194
max         5.102083
Name: days_since_last_login, dtype: float64


In [36]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Scale features - Isolation Forest doesn't strictly require this, 
# but it helps when features are on very different scales (hours vs counts)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# contamination = expected proportion of anomalies in the data
# we injected ~3%, so we tell the model roughly what to expect
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.03,
    random_state=42
)

iso_forest.fit(X_scaled)

# predict returns -1 for anomaly, 1 for normal — convert to 1/0 to match our labels
raw_preds = iso_forest.predict(X_scaled)
features_df['iso_pred'] = np.where(raw_preds == -1, 1, 0)

# anomaly_score: lower (more negative) = more anomalous
features_df['iso_score'] = iso_forest.decision_function(X_scaled)

print(features_df['iso_pred'].value_counts())

iso_pred
0    2817
1      88
Name: count, dtype: int64


In [37]:
# grab one of our injected volume_spike anomalies and one random normal session
# compare their iso_score directly - the anomaly should have a noticeably LOWER (more negative) score
sample_anomaly_idx = features_df[features_df['anomaly_type'] == 'volume_spike'].index[0]
sample_normal_idx = features_df[features_df['is_anomaly'] == 0].index[0]

print("Anomaly (volume_spike) score:", features_df.loc[sample_anomaly_idx, 'iso_score'])
print("Normal session score:", features_df.loc[sample_normal_idx, 'iso_score'])

Anomaly (volume_spike) score: -0.09288759862393259
Normal session score: 0.04959034498729464


In [38]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("Confusion Matrix:")
print(confusion_matrix(features_df['is_anomaly'], features_df['iso_pred']))

print("\nClassification Report:")
print(classification_report(features_df['is_anomaly'], features_df['iso_pred']))

print(f"\nROC-AUC (using anomaly score): {roc_auc_score(features_df['is_anomaly'], -features_df['iso_score']):.3f}")

Confusion Matrix:
[[2789   29]
 [  28   59]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      2818
           1       0.67      0.68      0.67        87

    accuracy                           0.98      2905
   macro avg       0.83      0.83      0.83      2905
weighted avg       0.98      0.98      0.98      2905


ROC-AUC (using anomaly score): 0.985


In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42, stratify=y
)

log_reg = LogisticRegression(class_weight='balanced', random_state=42)
log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_test)
y_proba = log_reg.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print(f"\nROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")

Confusion Matrix:
[[685  20]
 [  0  22]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.99       705
           1       0.52      1.00      0.69        22

    accuracy                           0.97       727
   macro avg       0.76      0.99      0.84       727
weighted avg       0.99      0.97      0.98       727


ROC-AUC: 0.996


In [40]:
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': log_reg.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

print(importance_df)

                  feature  coefficient
0              login_hour    -2.085077
2               is_new_ip     1.660536
6  high_sensitivity_count     1.255128
4       file_access_count     1.196583
5    distinct_files_count     1.195002
7            delete_count    -0.754616
1            is_off_hours     0.143836
3   days_since_last_login     0.131927


In [41]:
comparison = features_df.groupby('iso_pred')[feature_cols].mean().T
    
comparison.columns = ['normal_avg', 'anomaly_avg']
comparison['difference'] = (comparison['anomaly_avg'] - comparison['normal_avg']).abs()
comparison = comparison.sort_values('difference', ascending=False)

print(comparison)

                        normal_avg  anomaly_avg  difference
file_access_count         3.983316    12.454545    8.471230
distinct_files_count      3.966986    12.204545    8.237559
login_hour                8.463259     6.761364    1.701895
high_sensitivity_count    0.664182     1.681818    1.017636
is_off_hours              0.078452     0.488636    0.410184
days_since_last_login     1.537015     1.813621    0.276605
is_new_ip                 0.041889     0.272727    0.230839
delete_count              0.659922     0.886364    0.226442


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data=features_df, x='iso_score', hue='is_anomaly', bins=50, ax=ax, palette=['steelblue', 'crimson'])
ax.set_title('Isolation Forest Anomaly Scores: True Normal vs True Anomaly')
ax.set_xlabel('Anomaly Score (lower = more anomalous)')
plt.savefig('iso_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
importance_df_sorted = importance_df.sort_values('coefficient')
colors = ['crimson' if c > 0 else 'steelblue' for c in importance_df_sorted['coefficient']]
ax.barh(importance_df_sorted['feature'], importance_df_sorted['coefficient'], color=colors)
ax.set_title('Logistic Regression Feature Coefficients')
ax.set_xlabel('Coefficient (positive = pushes toward anomaly)')
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
off_hours_anomaly_rate = features_df.groupby('is_off_hours')['is_anomaly'].mean()
off_hours_anomaly_rate.plot(kind='bar', ax=ax, color=['steelblue', 'crimson'])
ax.set_title('Anomaly Rate: Off-Hours vs Normal-Hours Logins')
ax.set_xlabel('Is Off-Hours Login')
ax.set_ylabel('Anomaly Rate')
ax.set_xticklabels(['Normal Hours', 'Off Hours'], rotation=0)
plt.savefig('off_hours_anomaly_rate.png', dpi=150, bbox_inches='tight')
plt.show()